# 📁 Module 12 — File Handling (Text, CSV, JSON, Binary, SQLite)

## 🎯 What you will learn
1. Opening files: modes `r w a x b + t`, encodings
2. Reading (`read`, `readline`, `readlines`, iteration) and writing (`write`, `writelines`)
3. The `with` statement (always use it)
4. File pointer: `tell()` / `seek()`
5. `pathlib` for paths
6. CSV, JSON, pickle, binary files
7. Introduction to SQLite databases
8. Hidden facts & tricks

**Exam relevance:** ⭐⭐⭐⭐⭐ — file modes and programs (count words, copy file…) are standard.

> The examples create files in a scratch folder; they're safe to delete.

---
## 1. Why files?
Variables vanish when the program ends. Files give **persistent storage**.

## 2. `open()` and modes

`open(file, mode="r", encoding=None)`

| Mode | Meaning | If file missing | Existing content |
|---|---|---|---|
| `r` | read (default) | ❌ FileNotFoundError | kept |
| `w` | write | created | **erased** (truncated) |
| `a` | append | created | kept; writes go to end |
| `x` | exclusive create | created | ❌ FileExistsError if exists |
| `r+` | read + write | ❌ error | kept |
| `w+` | write + read | created | erased |
| `a+` | append + read | created | kept |
| `b` | binary (add to any: `rb`, `wb`) | | |
| `t` | text (default) | | |

In [1]:
f = open("hello.txt", "w", encoding="utf-8")
f.write("Hello, file!\n")
f.write("Second line\n")
f.close()                    # ALWAYS close (flushes buffers, frees the OS handle)

f = open("hello.txt")        # default mode "r"
print(f.read())
f.close()
print(f.closed)

Hello, file!
Second line

True


## 3. The `with` statement (best practice)
It closes the file automatically — even if an error occurs.

In [2]:
with open("poem.txt", "w") as f:
    f.write("Roses are red,\n")
    f.write("Violets are blue,\n")
    f.writelines(["Python is fun,\n", "And so are you.\n"])     # takes a list of strings (no auto \n)
print("closed?", f.closed)

closed? True


## 4. Reading — 4 ways

In [3]:
with open("poem.txt") as f:
    print(repr(f.read()))               # 1) whole file as ONE string

with open("poem.txt") as f:
    print(repr(f.read(10)))             # first 10 characters
    print(repr(f.read(5)))              # next 5 (pointer moves)

with open("poem.txt") as f:
    print(repr(f.readline()))           # 2) one line (includes \n)
    print(repr(f.readline()))

with open("poem.txt") as f:
    lines = f.readlines()               # 3) list of all lines
print(lines)

with open("poem.txt") as f:
    for line in f:                      # 4) iterate: BEST for big files (lazy, 1 line at a time)
        print(line.rstrip("\n"))

'Roses are red,\nViolets are blue,\nPython is fun,\nAnd so are you.\n'
'Roses are '
'red,\n'
'Roses are red,\n'
'Violets are blue,\n'
['Roses are red,\n', 'Violets are blue,\n', 'Python is fun,\n', 'And so are you.\n']
Roses are red,
Violets are blue,
Python is fun,
And so are you.


## 5. Appending & the modes in action

In [4]:
with open("log.txt", "w") as f:
    f.write("start\n")
with open("log.txt", "a") as f:         # append keeps old content
    f.write("second\n")
with open("log.txt", "w") as f:         # 'w' wipes it again!
    f.write("fresh\n")
print(open("log.txt").read())

try:
    open("log.txt", "x")                # exclusive creation
except FileExistsError as e:
    print("FileExistsError:", e)

with open("both.txt", "w+") as f:       # write then read back
    f.write("abcdef")
    f.seek(0)                           # rewind the pointer
    print(f.read())

fresh

FileExistsError: [Errno 17] File exists: 'log.txt'
abcdef


## 6. File pointer: `tell()` and `seek()`

- `f.tell()` → current position (bytes/chars from start)
- `f.seek(offset, whence)` → move; `whence` 0=start, 1=current, 2=end (relative seeks from current/end need binary mode)

In [5]:
with open("seek.txt", "w+") as f:
    f.write("0123456789")
    f.seek(3)
    print(f.tell(), f.read(2), f.tell())
    f.seek(0)
    print(f.read(4))

with open("seek.txt", "rb") as f:
    f.seek(-3, 2)                      # 3 bytes before the end
    print(f.read())

3 34 5
0123
b'789'


## 7. Classic file programs

In [6]:
# Create a sample file
text = """Python is easy to learn.
Python is powerful.
Files are useful.
"""
with open("sample.txt", "w") as f:
    f.write(text)

# 1. Count lines, words, characters
lines = words = chars = 0
with open("sample.txt") as f:
    for line in f:
        lines += 1
        words += len(line.split())
        chars += len(line)
print(lines, words, chars)

# 2. Word frequency
from collections import Counter
with open("sample.txt") as f:
    freq = Counter(w.strip(".,").lower() for line in f for w in line.split())
print(freq.most_common(3))

# 3. Copy a file
with open("sample.txt") as src, open("copy.txt", "w") as dst:
    for line in src:
        dst.write(line)

# 4. Copy, transforming (uppercase)
with open("sample.txt") as src, open("upper.txt", "w") as dst:
    dst.writelines(line.upper() for line in src)
print(open("upper.txt").read())

# 5. Longest line, last N lines
with open("sample.txt") as f:
    all_lines = f.read().splitlines()
print(max(all_lines, key=len), "|", all_lines[-2:])

# 6. Search for a word and show line numbers
with open("sample.txt") as f:
    for no, line in enumerate(f, 1):
        if "Python" in line:
            print(no, line.strip())

# 7. Append a timestamped entry
from datetime import datetime
with open("diary.txt", "a") as f:
    f.write(f"{datetime(2025,1,1,9,0):%Y-%m-%d %H:%M} - started learning Python\n")
print(open("diary.txt").read().strip())

# 8. Merge two files, and remove blank lines
with open("a.txt", "w") as f: f.write("A1\n\nA2\n")
with open("b.txt", "w") as f: f.write("B1\n")
with open("merged.txt", "w") as out:
    for name in ("a.txt", "b.txt"):
        with open(name) as f:
            out.writelines(l for l in f if l.strip())
print(open("merged.txt").read().split())

3 11 63
[('python', 2), ('is', 2), ('easy', 1)]
PYTHON IS EASY TO LEARN.
PYTHON IS POWERFUL.
FILES ARE USEFUL.

Python is easy to learn. | ['Python is powerful.', 'Files are useful.']
1 Python is easy to learn.
2 Python is powerful.
2025-01-01 09:00 - started learning Python
['A1', 'A2', 'B1']


## 8. `pathlib` & `os` — working with paths and folders

In [7]:
from pathlib import Path
import os

base = Path("project"); base.mkdir(exist_ok=True)
(base / "notes.txt").write_text("hello\nworld\n")
(base / "data").mkdir(exist_ok=True)
(base / "data" / "x.csv").write_text("a,b\n1,2\n")

print((base / "notes.txt").read_text().splitlines())
print([p.name for p in base.iterdir()])
print([str(p) for p in base.rglob("*.csv")])          # recursive search
p = base / "notes.txt"
print(p.name, p.stem, p.suffix, p.parent, p.is_file(), p.is_dir(), p.stat().st_size)
p.rename(base / "renamed.txt")
print(sorted(os.listdir(base)))
(base / "renamed.txt").unlink()                        # delete a file
import shutil; shutil.rmtree(base)                     # delete a folder tree
print(base.exists())

['hello', 'world']
['data', 'notes.txt']
['project\\data\\x.csv']
notes.txt notes .txt project True False 14
['data', 'renamed.txt']
False


## 9. CSV files

In [8]:
import csv
rows = [["name", "dept", "salary"],
        ["Asha", "IT", 50000],
        ["Ravi", "HR", 42000],
        ["Meena", "IT", 61000]]

with open("staff.csv", "w", newline="") as f:          # newline="" avoids blank rows on Windows
    csv.writer(f).writerows(rows)

with open("staff.csv", newline="") as f:
    for row in csv.reader(f):
        print(row)

with open("staff.csv", newline="") as f:
    people = list(csv.DictReader(f))                   # each row -> dict using header
print(people[0], people[0]["name"])
print("IT total:", sum(int(p["salary"]) for p in people if p["dept"] == "IT"))

with open("out.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["name", "salary"], extrasaction="ignore")
    w.writeheader()
    w.writerows(people)
print(open("out.csv").read())

['name', 'dept', 'salary']
['Asha', 'IT', '50000']
['Ravi', 'HR', '42000']
['Meena', 'IT', '61000']
{'name': 'Asha', 'dept': 'IT', 'salary': '50000'} Asha
IT total: 111000
name,salary
Asha,50000
Ravi,42000
Meena,61000



## 10. JSON files

In [9]:
import json
data = {"course": "Python", "students": [{"name": "Asha", "marks": 90}, {"name": "Ravi", "marks": 80}], "open": True}

with open("data.json", "w") as f:
    json.dump(data, f, indent=2)                       # object -> file

with open("data.json") as f:
    loaded = json.load(f)                              # file -> object
print(loaded["students"][1]["name"], loaded == data)

s = json.dumps(data, sort_keys=True)                   # object -> string
print(s[:60])
print(json.loads('{"x": [1, 2.5, null, true]}'))       # null->None, true->True

Ravi True
{"course": "Python", "open": true, "students": [{"marks": 90
{'x': [1, 2.5, None, True]}


> JSON maps: object↔dict, array↔list, string↔str, number↔int/float, true/false↔True/False, null↔None. It can't store tuples (become lists), sets, or custom classes without help.

## 11. Binary files & pickle

In [10]:
# Binary write/read
data = bytes([72, 101, 108, 108, 111, 0, 255])
with open("blob.bin", "wb") as f:
    f.write(data)
with open("blob.bin", "rb") as f:
    content = f.read()
print(content, list(content), len(content))

# Copy any file (images, PDFs...) in chunks
with open("blob.bin", "rb") as src, open("blob_copy.bin", "wb") as dst:
    while chunk := src.read(4):
        dst.write(chunk)

# pickle: serialise almost any Python object (only load pickles you trust!)
import pickle
obj = {"scores": [1, 2, 3], "tuple": (1, 2), "set": {1, 2}}
with open("obj.pkl", "wb") as f:
    pickle.dump(obj, f)
with open("obj.pkl", "rb") as f:
    print(pickle.load(f))

b'Hello\x00\xff' [72, 101, 108, 108, 111, 0, 255] 7
{'scores': [1, 2, 3], 'tuple': (1, 2), 'set': {1, 2}}


## 12. Encoding & text pitfalls

In [11]:
with open("uni.txt", "w", encoding="utf-8") as f:
    f.write("café – 你好 – 🐍")
print(open("uni.txt", encoding="utf-8").read())
print(open("uni.txt", "rb").read())
try:
    open("uni.txt", encoding="ascii").read()
except UnicodeDecodeError as e:
    print("UnicodeDecodeError:", e.reason)

# Windows line endings: text mode translates \r\n <-> \n for you.
# ALWAYS pass encoding="utf-8" explicitly for portability.

café – 你好 – 🐍
b'caf\xc3\xa9 \xe2\x80\x93 \xe4\xbd\xa0\xe5\xa5\xbd \xe2\x80\x93 \xf0\x9f\x90\x8d'
UnicodeDecodeError: ordinal not in range(128)


## 13. Databases: a taste of SQLite (`sqlite3`, built in)
SQLite is a full SQL database in a single file. Perfect for small apps.

In [12]:
import sqlite3
con = sqlite3.connect(":memory:")              # use "app.db" for a real file
cur = con.cursor()
cur.execute("CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT NOT NULL, marks REAL)")
cur.executemany("INSERT INTO students (name, marks) VALUES (?, ?)",   # ? placeholders prevent SQL injection!
                [("Asha", 91.5), ("Ravi", 78.0), ("Meena", 85.25)])
con.commit()

for row in cur.execute("SELECT id, name, marks FROM students WHERE marks > ? ORDER BY marks DESC", (80,)):
    print(row)
print(cur.execute("SELECT AVG(marks), COUNT(*) FROM students").fetchone())

con.row_factory = sqlite3.Row
r = con.execute("SELECT * FROM students WHERE name = ?", ("Ravi",)).fetchone()
print(dict(r))
con.close()

(1, 'Asha', 91.5)
(3, 'Meena', 85.25)
(84.91666666666667, 3)
{'id': 2, 'name': 'Ravi', 'marks': 78.0}


---
# 🕵️ Hidden Facts & Pro Tricks

In [13]:
# 1. Read a file into a list of stripped lines in one line
lines = open("sample.txt").read().splitlines()
print(lines)

# 2. Path.read_text / write_text: one-liners, no with needed
from pathlib import Path
Path("quick.txt").write_text("fast\n")
print(Path("quick.txt").read_text())

# 3. print() can write to a file
with open("printed.txt", "w") as f:
    print("via print", 42, file=f)
print(open("printed.txt").read().strip())

# 4. Temporary files clean themselves up
import tempfile
with tempfile.TemporaryDirectory() as tmp:
    (Path(tmp) / "t.txt").write_text("temp")
    print(len(list(Path(tmp).iterdir())))
print(Path(tmp).exists())

# 5. Atomic-safe write: write to temp then replace (never leaves half-written files)
import os
Path("config.tmp").write_text("v=2")
os.replace("config.tmp", "config.txt")
print(Path("config.txt").read_text())

# 6. Files are iterators -> next(f) reads a line; read the header then the rest
with open("staff.csv") as f:
    header = next(f).strip().split(",")
    body = [line.strip().split(",") for line in f]
print(header, len(body))

# 7. Reading huge files: iterate, or mmap; never f.read() a 10 GB file
# 8. gzip/zipfile/tarfile work like open()
import gzip
with gzip.open("big.txt.gz", "wt") as f: f.write("compress me " * 100)
with gzip.open("big.txt.gz", "rt") as f: print(len(f.read()))
import zipfile
with zipfile.ZipFile("pack.zip", "w") as z: z.write("sample.txt")
print(zipfile.ZipFile("pack.zip").namelist())

# 9. file objects have handy attributes
with open("sample.txt", "r+") as f:
    print(f.name, f.mode, f.encoding, f.readable(), f.writable(), f.seekable())

# 10. Universal glob: find all .py files recursively -> list(Path(".").rglob("*.py"))
# 11. os.walk visits every folder; shutil.make_archive zips a folder in one call
# 12. watch out: open("x", "w") truncates IMMEDIATELY when opened, before you write anything!

['Python is easy to learn.', 'Python is powerful.', 'Files are useful.']
fast

via print 42
1
False
v=2
['name', 'dept', 'salary'] 3
1200
['sample.txt']
sample.txt r+ cp1252 True True True


**Hidden fact:** Python buffers file writes. If your program crashes before `close()`/`flush()`, data may be lost — one more reason to use `with`. `f.flush()` pushes buffers to the OS; `os.fsync(f.fileno())` pushes to disk.

**Hidden fact:** on Windows, `\` in normal strings is an escape char: `"C:\new\test.txt"` contains a newline and a tab! Use raw strings `r"C:\new\test.txt"`, forward slashes `"C:/new/test.txt"`, or `Path`.

---
# 📝 Exam Corner

**Theory**
1. List file modes and explain `r`, `w`, `a`, `r+`, `w+`, `rb`.
2. Difference between `read()`, `readline()`, `readlines()`.
3. Why use `with open(...)`?
4. Explain `seek()` and `tell()`.
5. Difference between text and binary files.
6. What is serialization? `pickle` vs `json`.

**Programs to practise (write from memory!)**
- Read a file and count vowels / lines / words
- Copy contents of one file to another; reverse the lines
- Append user data to a file; search a word
- Read CSV and compute totals; write dict to JSON

**Output prediction**

In [14]:
with open("q.txt", "w") as f:
    f.write("one\ntwo\nthree\n")
with open("q.txt") as f:
    print(f.readline().strip(), f.read(3), repr(f.readline()))
with open("q.txt") as f:
    print(len(f.readlines()), f.read() == "")

one two '\n'
3 True


<details><summary>💡 Answers</summary>

```
one two '\n'
3 True
```
`readline()` → "one", `read(3)` → "two", `readline()` → the rest of that line `'\n'`. After `readlines()` the pointer is at the end so `read()` returns `""`.

> 🪟 **Windows note:** files written in text mode store `\r\n`, so `tell()` counts 2 bytes per line break there (10 instead of 8 for this file). Byte positions differ between operating systems — exams only ask about the logic.
</details>

---
## ✅ Summary
- `open()` modes: `r` `w` `a` `x` `+` `b`. `w` erases! Always `with`.
- Read: `read()`, `readline()`, `readlines()`, iterate lines. Write: `write()`, `writelines()`.
- `seek()/tell()` move/report the pointer. `pathlib` > `os.path`.
- `csv`, `json`, `pickle`, `sqlite3` cover 95% of storage needs.

➡️ **Next:** Module 13 — Object-Oriented Programming (Part 1).